# Quiz Solutions — Magnetic Resonance Image Formation

The quiz asks to *answer any 5 for 50 points*. For completeness, **all 7 questions** are answered
here, with short illustrative plots where helpful.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Q1. How is a proton different from an iron filing in a magnetic field?

An iron filing is a **classical ferromagnetic dipole**: in a field it simply experiences a torque
and **aligns** with the field lines, then stops. A proton has **intrinsic spin angular momentum**
$\mathbf{L}$ *and* a **magnetic moment** $\boldsymbol{\mu}$. The two “key properties” are linked by the
**gyromagnetic ratio**
$$\gamma = \frac{\mu}{L}\quad(\boldsymbol{\mu}=\gamma\mathbf{L}).$$

Because the proton carries angular momentum, a torque does not just align it — it makes it
**precess** about the field at the Larmor frequency $\omega_0=\gamma B_0$ (like a spinning top),
rather than simply snapping into alignment. That precessing magnetic moment is what makes NMR/MRI
possible; an iron filing has no such spin-locked precession.

## Q2. What principle do the RF coils work on, and where does the FID come from?

**Principle: Faraday’s law of electromagnetic induction.** A time-varying magnetic flux through the
receive coil induces an EMF (voltage).

**Where the FID comes from:** in the static field $B_0$ the net magnetization is along $z$ and
produces no signal. An RF pulse tips $\mathbf{M}$ into the transverse plane; the resulting
**transverse magnetization $M_{xy}$ precesses at $\omega_0$**, creating an oscillating flux through
the coil. By Faraday’s law this induces an oscillating voltage — the **Free Induction Decay (FID)** —
which decays as the spins lose transverse coherence.

## Q3. Why does the FID occur, when would it take “forever” to decay, and the disturbed case

**Why it occurs / decays:** the FID is the precessing transverse magnetization. It **decays because
spins dephase** (lose phase coherence) — from true spin–spin relaxation ($T_2$) plus static field
inhomogeneity (together $T_2^*$).

**When it would take infinitely long to decay:** if there were **no dephasing** — a perfectly
homogeneous $B_0$ and no spin–spin relaxation ($T_2^*\to\infty$). Then $M_{xy}$ precesses forever as an
**undamped sinusoid**.

**Hypothetical disturbance:** if a strong external magnet perturbs $B_0$ mid-sequence, the Larmor
frequency **jumps abruptly** and the spins dephase very rapidly, so the FID shows a sudden change in
frequency and a fast collapse of amplitude at the moment of disturbance (sketched below).

In [ ]:
t = np.linspace(0, 0.3, 4000)
f1 = 60.0            # FID (demo) frequency [Hz]
T2star = 0.05        # transverse dephasing time [s]

fid        = np.exp(-t / T2star) * np.cos(2 * np.pi * f1 * t)
undamped   = np.cos(2 * np.pi * f1 * t)                       # T2* -> infinity

# Hypothetical: B0 disturbed at t_d -> frequency jump + rapid dephasing
t_d, f2, T2star_fast = 0.12, 110.0, 0.008
fid_dist = fid.copy()
m = t >= t_d
A_td   = np.exp(-t_d / T2star)
phase0 = 2 * np.pi * f1 * t_d
fid_dist[m] = A_td * np.exp(-(t[m] - t_d) / T2star_fast) * np.cos(2 * np.pi * f2 * (t[m] - t_d) + phase0)

fig, ax = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
ax[0].plot(t, fid, color="tab:blue");   ax[0].set_title("Normal FID (decays with T2*)")
ax[1].plot(t, undamped, color="tab:green"); ax[1].set_title(r"Hypothetical $T_2^*\to\infty$: never decays")
ax[2].plot(t, fid_dist, color="tab:red")
ax[2].axvline(t_d, color="k", ls="--", lw=0.8)
ax[2].set_title("B0 disturbed mid-FID: frequency jump + fast dephasing")
ax[2].set_xlabel("time [s]")
for a_ in ax:
    a_.set_ylabel("coil signal"); a_.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Q4. Which components of M are measured, their phase, and the longitudinal recovery constant

With $B_0$ along $z$, only the **transverse components $M_x$ and $M_y$** (together $M_{xy}$) induce
signal in the coil — these are what is measured. $M_z$ (longitudinal) does not directly induce signal.

- $M_x$ and $M_y$ are **90° out of phase** (quadrature): one is a cosine, the other a sine of
  $\omega_0 t$.
- Both transverse components **decay with $T_2$**.
- $M_z$ **recovers** toward $M_0$ with time constant **$T_1$** (spin–lattice), as
  $M_z=M_0(1-e^{-t/T_1})$.

In [ ]:
t = np.linspace(0, 2.0, 2000)
T1, T2, f = 0.8, 0.3, 3.0

Mx = np.exp(-t / T2) * np.cos(2 * np.pi * f * t)
My = np.exp(-t / T2) * np.sin(2 * np.pi * f * t)   # 90 deg out of phase with Mx
Mz = 1 - np.exp(-t / T1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, Mx, label="Mx (~cos, decays with T2)")
ax.plot(t, My, label="My (~sin, 90 deg out of phase)")
ax.plot(t, Mz, label="Mz (recovers with T1)", lw=2, color="k")
ax.set_xlabel("time [s]"); ax.set_ylabel("magnetization (a.u.)")
ax.set_title("Transverse (T2) vs longitudinal (T1) components")
ax.legend(loc="center right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Q5. The FID solves which equations, and which rate governs transverse decay?

The FID of a precessing proton is the solution to the **Bloch equations**.

The relaxation constant governing the **permanent decay of FID amplitude in the plane normal to
$B_0$** (the transverse plane) is **$T_2$** — the **spin–spin (transverse) relaxation** time. (Field
inhomogeneity adds reversible dephasing to give the observed $T_2^*\le T_2$, but the *irreversible*
part is $T_2$.)

## Q6. Resonance frequency vs field strength, and slice selection

The resonance (precession) frequency is proportional to the field via the **Larmor equation**:
$$\omega_0 = \gamma B_0 \qquad\text{or}\qquad f_0 = \frac{\gamma}{2\pi} B_0.$$

**Slice selection:** apply a **longitudinal gradient $G_z$** so the field (and frequency) varies
along $z$:
$$f(z) = \frac{\gamma}{2\pi}\,(B_0 + G_z z).$$
Now each $z$ resonates at a unique frequency, so transmitting an RF pulse containing a **band of
frequencies** $f_0\pm BW/2$ excites only the slab whose $f(z)$ falls in that band. Slice thickness
$\Delta z = BW/[(\gamma/2\pi)G_z]$; slice position is set by the center frequency $f_0$.

In [ ]:
gamma_bar = 42.577e6         # Hz/T
B0, Gz, z0, dz = 1.5, 10e-3, 0.0, 5e-3
z = np.linspace(-0.1, 0.1, 500)
f_z = gamma_bar * (B0 + Gz * z)
BW = gamma_bar * Gz * dz

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(z * 1e3, f_z / 1e6, color="tab:blue")
ax.axvspan((z0 - dz / 2) * 1e3, (z0 + dz / 2) * 1e3, color="tab:orange", alpha=0.35,
           label=f"selected slice ({dz*1e3:.0f} mm, BW={BW:.0f} Hz)")
ax.set_xlabel("z [mm]"); ax.set_ylabel("f(z) [MHz]")
ax.set_title("Slice selection: frequency is linear in z under a gradient")
ax.legend(); plt.tight_layout(); plt.show()

## Q7. Damadian (1971): which relaxation rate distinguished tumor, and how the two differ

In Damadian, R. (1971), *Tumor detection by nuclear magnetic resonance* (Science, 171), the two
relaxation times measured were:
- **$T_1$ — spin–lattice (longitudinal) relaxation**, and
- **$T_2$ — spin–spin (transverse) relaxation**.

The rate found **statistically significant in distinguishing tumor tissue was $T_1$ (spin–lattice)**:
tumors showed markedly **prolonged $T_1$** (and also elevated $T_2$) relative to normal tissue.

**How they differ in terms of spin energy:**
- **$T_1$ (spin–lattice):** the excited spins give up energy **to the surrounding lattice**
  (environment) as $M_z$ recovers — a true loss of spin energy to the surroundings.
- **$T_2$ (spin–spin):** spins lose **phase coherence** with each other through mutual spin
  interactions, **without net energy loss to the lattice** — it is dephasing, not energy dissipation.

In [ ]:
# Representative relaxation times (seconds) from the relaxometry tables
tissues = ["normal liver", "Novikoff hepatoma (tumor)"]
T1_vals = [0.293, 0.826]
T2_vals = [0.050, 0.118]

x = np.arange(len(tissues)); w = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - w / 2, T1_vals, w, label="T1 (spin-lattice)", color="tab:blue")
ax.bar(x + w / 2, T2_vals, w, label="T2 (spin-spin)", color="tab:orange")
ax.set_xticks(x); ax.set_xticklabels(tissues)
ax.set_ylabel("relaxation time [s]")
ax.set_title("Tumor vs normal: prolonged T1 (and T2) in tumor")
ax.legend(); plt.tight_layout(); plt.show()

print(f"T1 ratio tumor/normal = {T1_vals[1]/T1_vals[0]:.2f}x")
print(f"T2 ratio tumor/normal = {T2_vals[1]/T2_vals[0]:.2f}x")

## Quick-reference summary
| Q | Answer |
|---|--------|
| 1 | Proton has spin angular momentum **and** magnetic moment (linked by $\gamma=\mu/L$) → it **precesses**; an iron filing just aligns. |
| 2 | **Faraday’s law** of induction; FID = precessing transverse $M_{xy}$ inducing coil voltage after the RF pulse. |
| 3 | FID decays by dephasing ($T_2^*$); lasts forever if $T_2^*\to\infty$ (homogeneous field, no relaxation); a mid-pulse field disturbance causes a frequency jump + rapid collapse. |
| 4 | Measured: **$M_x, M_y$** (90° out of phase, decay with $T_2$); $M_z$ recovers with **$T_1$**. |
| 5 | **Bloch** equations; transverse decay governed by **$T_2$** (spin–spin). |
| 6 | **Larmor equation** $\omega_0=\gamma B_0$; with gradient $G_z$, $f(z)$ is linear in $z$ → frequency band selects a slice. |
| 7 | **$T_1$ (spin–lattice)** most significant; $T_1$ = energy loss to lattice, $T_2$ = loss of phase coherence (no net energy loss). |